# Domain-Adaptive RAG with Fine-Tuned Embeddings
## Demo Notebook

This notebook demonstrates the full pipeline:
1. Load a query
2. Compare retrieval: BM25 vs baseline embedding vs fine-tuned embedding  
3. Show final evaluation results
4. Discuss findings including out-of-distribution degradation

**Dataset:** 5,000 arXiv CS abstracts (2021)  
**Task:** Given a query, retrieve the most relevant abstract chunk

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))  # so we can import from project root

import faiss
import pickle
import numpy as np
import json
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from baseline_rag import embed_text as baseline_embed
from config import BASELINE_DIR, FINETUNED_DIR, FINETUNED_MODEL_PATH, TOP_K

# Load baseline index
b_index = faiss.read_index(f"{BASELINE_DIR}/baseline.faiss")
with open(f"{BASELINE_DIR}/baseline_chunks.pkl", "rb") as f:
    b_chunks = pickle.load(f)

# Load finetuned index
ft_index = faiss.read_index(f"{FINETUNED_DIR}/finetuned.faiss")
with open(f"{FINETUNED_DIR}/finetuned_chunks.pkl", "rb") as f:
    ft_chunks = pickle.load(f)

# Load finetuned model
ft_model = SentenceTransformer(FINETUNED_MODEL_PATH)
ft_embed = lambda t: ft_model.encode(t, normalize_embeddings=True).tolist()

# Build BM25 index
tokenized_corpus = [c["text"].lower().split() for c in b_chunks]
bm25 = BM25Okapi(tokenized_corpus)

print("All indexes loaded successfully.")
print(f"Corpus: {len(b_chunks)} chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

All indexes loaded successfully.
Corpus: 13050 chunks


## Example 1 — In-distribution query
A query typical of our training distribution (arXiv physics/CS style)

In [2]:
def compare_retrieval(query, top_k=3):
    """Show top-1 result from all three retrieval methods side by side."""
    print(f"Query: '{query}'")
    print("=" * 70)

    # BM25
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_bm25 = b_chunks[np.argsort(scores)[::-1][0]]["text"]
    print(f"\nBM25 (keyword):")
    print(f"  {top_bm25[:300]}")

    # Baseline embedding
    q_base = np.array([baseline_embed(query)], dtype="float32")
    faiss.normalize_L2(q_base)
    _, b_idxs = b_index.search(q_base, top_k)
    print(f"\nBaseline embedding (nomic-embed-text):")
    print(f"  {b_chunks[b_idxs[0][0]]['text'][:300]}")

    # Fine-tuned embedding
    q_ft = ft_model.encode([query], normalize_embeddings=True).astype("float32")
    _, ft_idxs = ft_index.search(q_ft, top_k)
    print(f"\nFine-tuned embedding (ours):")
    print(f"  {ft_chunks[ft_idxs[0][0]]['text'][:300]}")
    print("=" * 70)

# In-distribution query — matches training domain
compare_retrieval("What is the convergence rate of gradient descent for convex functions?")

Query: 'What is the convergence rate of gradient descent for convex functions?'

BM25 (keyword):
  We provide a sharp rate of convergence in the central limit theorem for random vectors with an unconditional, log-concave density. The argument relies on analysis of the Neumann laplacian on convex domains and on the theory of optimal transportation of measures.

Baseline embedding (nomic-embed-text):
  We study Markov processes associated with stochastic differential equations, whose non-linearities are gradients of convex functionals. We prove a general result of existence of such Markov processes and a priori estimates on the transition probabilities. The main result is the following stability p

Fine-tuned embedding (ours):
  int, as a constrained convex optimization problem to which we apply semidefinite programming techniques. We show by an extensive numerical study that the optimal design improves significantly the convergence speed of the consensus algorithm and can achieve the as

## Example 2 — Another in-distribution query

In [3]:
compare_retrieval("How are hidden Markov models applied to sequence labeling problems?")

Query: 'How are hidden Markov models applied to sequence labeling problems?'

BM25 (keyword):
  In this paper, we introduce the on-line Viterbi algorithm for decoding hidden Markov models (HMMs) in much smaller than linear space. Our analysis on two-state HMMs suggests that the expected maximum memory used to decode sequence of length with -state HMM can be as low as , without a significant sl

Baseline embedding (nomic-embed-text):
  Motivation: Profile hidden Markov Models (pHMMs) are a popular and very useful tool in the detection of the remote homologue protein families. Unfortunately, their performance is not always satisfactory when proteins are in the 'twilight zone'. We present HMMER-STRUCT, a model construction algorithm

Fine-tuned embedding (ours):
  Motivation: Profile hidden Markov Models (pHMMs) are a popular and very useful tool in the detection of the remote homologue protein families. Unfortunately, their performance is not always satisfactory when proteins are in the 

## Example 3 — Out-of-distribution query
A general CS query **not** represented in our training corpus.  
This demonstrates catastrophic forgetting in the fine-tuned model.

In [4]:
compare_retrieval("How do transformer attention mechanisms work?")

Query: 'How do transformer attention mechanisms work?'

BM25 (keyword):
  Simulations of physicists for the competition between adult languages since 2003 are reviewed. How many languages are spoken by how many people? How many languages are contained in various language families? How do language similarities decay with geographical distance, and what effects do natural b

Baseline embedding (nomic-embed-text):
  A fundamental problem in neuroscience is understanding how working memory -- the ability to store information at intermediate timescales, like 10s of seconds -- is implemented in realistic neuronal networks. The most likely candidate mechanism is the attractor network, and a great deal of effort has

Fine-tuned embedding (ours):
  A fundamental problem in neuroscience is understanding how working memory -- the ability to store information at intermediate timescales, like 10s of seconds -- is implemented in realistic neuronal networks. The most likely candidate mechanism is the

## Evaluation Results

Evaluated on 100 held-out queries never seen during training.  
Retrieval metric: Hit Rate@3 and MRR@3 (no LLM judge — pure retrieval quality).

In [5]:
with open(f"{FINETUNED_DIR}/comparison_scores.json") as f:
    scores = json.load(f)

print(f"\n{'Method':<35} {'Hit Rate@3':>12} {'MRR@3':>10}")
print("-" * 60)
for method, s in sorted(scores.items(), key=lambda x: x[1]["hit_rate"]):
    marker = "  ◀ ours" if "Fine-tuned" in method else ""
    print(f"  {method:<35} {s['hit_rate']:>12.4f} {s['mrr']:>10.4f}{marker}")

base = scores["Baseline (nomic-embed-text)"]
ft   = scores["Fine-tuned (ours)"]
print(f"\nImprovement over baseline:")
print(f"  Hit Rate: {ft['hit_rate'] - base['hit_rate']:+.4f}  ({((ft['hit_rate']-base['hit_rate'])/base['hit_rate'])*100:+.1f}%)")
print(f"  MRR:      {ft['mrr'] - base['mrr']:+.4f}  ({((ft['mrr']-base['mrr'])/base['mrr'])*100:+.1f}%)")


Method                                Hit Rate@3      MRR@3
------------------------------------------------------------
  Baseline (nomic-embed-text)               0.6900     0.6117
  BM25                                      0.7000     0.6517
  Fine-tuned (ours)                         0.7100     0.6633  ◀ ours

Improvement over baseline:
  Hit Rate: +0.0200  (+2.9%)
  MRR:      +0.0516  (+8.4%)


## Out-of-Distribution Evaluation

In [6]:
with open(f"{FINETUNED_DIR}/ood_eval.json") as f:
    ood = json.load(f)

print(f"{'Query':<50} {'Baseline':>10} {'Fine-tuned':>12}")
print("-" * 75)
for q in ood["queries"]:
    marker = " ⚠" if q["finetuned"] < q["baseline"] else " ✓"
    print(f"  {q['query'][:48]:<50} {q['baseline']:>10.4f} {q['finetuned']:>12.4f}{marker}")
print("-" * 75)
print(f"  {'Mean':<50} {ood['baseline_mean_relevance']:>10.4f} {ood['finetuned_mean_relevance']:>12.4f}")
print(f"\n  OOD delta: {ood['delta']:+.4f}")

Query                                                Baseline   Fine-tuned
---------------------------------------------------------------------------
  How do transformer attention mechanisms work?          0.0556       0.0000 ⚠
  What are methods for training neural language mo       0.2000       0.1333 ⚠
  How does reinforcement learning handle explorati       0.0000       0.0000 ✓
  What is the role of convolutional layers in imag       0.2000       0.1333 ⚠
  How do graph neural networks propagate informati       0.3333       0.1333 ⚠
  What clustering algorithms work well for high di       0.0000       0.1667 ✓
  How does dropout prevent overfitting in deep net       0.0000       0.0000 ✓
  What are common approaches to machine translatio       0.2000       0.0000 ⚠
  How do generative adversarial networks produce r       0.0000       0.0000 ✓
  What optimization algorithms are used in deep le       0.1333       0.0667 ⚠
-----------------------------------------------------------

## Key Findings

### What worked
- Fine-tuning on 2,700 domain-specific query-passage pairs improved 
  in-distribution retrieval by **+8.4% MRR** over the off-the-shelf baseline
- Fine-tuned model (80MB, all-MiniLM-L6-v2) outperformed a larger 
  off-the-shelf model (nomic-embed-text) on domain queries

### What we observed
- **Catastrophic forgetting:** fine-tuned model shows -43.6% keyword 
  relevance on out-of-distribution general CS queries
- Both models score low on OOD queries, partly because the 2021 arXiv 
  corpus is sparse on topics like transformers and GANs

### What production systems do differently
1. **Mix general + domain pairs** during fine-tuning to preserve breadth
2. **Use a larger base model** (e.g. `bge-base-en-v1.5`) with more 
   capacity to retain general knowledge
3. **Continual learning** with regularization (EWC) to prevent forgetting
4. **Evaluate on both** in-distribution and OOD sets during training 
   and use OOD score as an early stopping signal